![image.png](https://i.imgur.com/a3uAqnb.png)

# Advanced QA Lab: Encoder, Decoder, or Both

- **Dataset**: short paragraphs and questions written in the notebook
- **Task**: question answering about a given paragraph
- **Models**: DistilBERT fine-tuned on SQuAD (encoder-only), FLAN-T5 small (encoder-decoder), Qwen3-0.6B (decoder-only), all pretrained and used without further training

In this lab, you'll explore three ways a transformer can answer a question about a paragraph:
- **Extractive QA** using BERT, an **encoder-only** model: it points at the answer inside the paragraph
- **Generative QA** using FLAN-T5, an **encoder-decoder** model: the encoder reads, the decoder writes the answer
- **Generative QA** using Qwen3, a **decoder-only** model: it reads the paragraph and the question as one text and continues it

You'll:
- See which tokens each architecture lets a token look at
- Analyze tokenization
- Run manual QA with all three models
- Investigate hallucinations
- Test what happens when the text is longer than a model's window
- Solve real-world examples

###📦 1. Setup


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    pipeline
)
import torch
import numpy as np

## Three Architectures, One Task

All three models are built from the same transformer blocks. What differs is which half of the transformer they keep, and therefore which tokens each token may look at:

| | encoder-only | encoder-decoder | decoder-only |
|---|---|---|---|
| model in this lab | DistilBERT fine-tuned on SQuAD | FLAN-T5 (small) | Qwen3 (0.6B) |
| each token looks at | every token, before and after it | encoder: every input token; decoder: earlier output tokens, plus the whole input through cross-attention | only the tokens before it |
| pretrained to | predict masked words | rebuild corrupted spans | predict the next token |
| how it answers | picks a start and an end token in the paragraph | writes an answer | continues the text with an answer |

Because BERT can only point at a span, it can never say anything that is not in the paragraph. T5 and Qwen3 write their answers, so they can say anything at all, including things that are not true.

## 🔍 How Pretrained QA Models Work

### 1. Extractive QA (BERT)
- Input: `question + context`
- Output: Start and end indices of answer **within the context**
- Model: `AutoModelForQuestionAnswering`

### 2. Generative QA (T5)
- Input: `"question: ... context: ..."`
- Output: A generated answer from the model
- Model: `AutoModelForSeq2SeqLM`

---

### 3. Generative QA (Qwen3, decoder-only)
- Input: one text, `context + question`, wrapped in the model's chat template
- Output: the tokens the model writes after it, one at a time
- Model: `AutoModelForCausalLM`

In [ ]:
bert_model_name = "distilbert-base-uncased-distilled-squad"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModelForQuestionAnswering.from_pretrained(bert_model_name, attn_implementation="eager")

t5_model_name = "google/flan-t5-small"
t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name, attn_implementation="eager")

## Loading the Decoder-Only Model

Qwen3-0.6B has 0.6 billion parameters, about nine times DistilBERT, so it goes on the GPU when there is one. `ask_qwen` builds the prompt with the model's chat template (with its step-by-step "thinking" mode switched off, so it answers directly), lets it write greedily, and returns only the newly written tokens.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

qwen_model_name = "Qwen/Qwen3-0.6B"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    attn_implementation="eager",   # the plain attention code, which can return attention weights
).to(device)


def ask_qwen(question, context=None, max_new_tokens=64):
    """Ask Qwen3 a question, optionally about a context paragraph; return its answer."""
    if context is None:
        text = question
    else:
        text = f"Context: {context}\nQuestion: {question}\nAnswer briefly, using only the context."
    messages = [{"role": "user", "content": text}]
    prompt = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = qwen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print(f"Qwen3 loaded on {device}: {sum(p.numel() for p in qwen_model.parameters()) / 1e6:.0f}M parameters")

## Who May Look at Whom

Each attention layer gives every token a weight for every other token, and each token's weights sum to 1. An encoder lets a token look at the tokens after it; a decoder must not, because when it writes it has not produced them yet. We can check this directly: run one sentence through each model, ask for the attention weights of every layer and head, and measure what share of each token's weight goes to **later** tokens (the entries above the diagonal of each attention matrix).

All three models were loaded with the plain ("eager") attention code, which can return its weights.

In [ ]:
sentence = "the movie was not bad"


def share_on_later_tokens(attentions):
    """Average share of a token's attention that goes to later tokens, over all layers, heads and tokens."""
    return torch.stack([layer[0].triu(1).sum(-1).mean() for layer in attentions]).mean().item()


with torch.no_grad():
    bert_out = bert_model(**bert_tokenizer(sentence, return_tensors="pt"), output_attentions=True)
    qwen_out = qwen_model(**qwen_tokenizer(sentence, return_tensors="pt").to(device), output_attentions=True)
    t5_in = t5_tokenizer("translate English to German: " + sentence, return_tensors="pt")
    t5_dec = t5_tokenizer("Der Film war", return_tensors="pt")
    t5_out = t5_model(**t5_in, decoder_input_ids=t5_dec["input_ids"], output_attentions=True)

print("Share of attention on later tokens")
print(f"  encoder-only (BERT):          {share_on_later_tokens(bert_out.attentions):.3f}")
print(f"  encoder-decoder (T5) encoder: {share_on_later_tokens(t5_out.encoder_attentions):.3f}")
print(f"  encoder-decoder (T5) decoder: {share_on_later_tokens(t5_out.decoder_attentions):.3f}")
print(f"  decoder-only (Qwen3):         {share_on_later_tokens(qwen_out.attentions):.3f}")
print("T5 cross-attention, (batch, heads, decoder tokens, encoder tokens):",
      tuple(t5_out.cross_attentions[0].shape))

### Exercise: Reading the Check
1. Which models give exactly zero weight to later tokens, and why must they?
2. The T5 cross-attention has one row per decoder token and one column per encoder token. What does a row tell you?

---

## 🔤 Tokenizer Internals

Understanding how text is split into tokens is crucial for QA models.

We’ll explore how BERT tokenizes a `question + context` pair and aligns it to the original text.

---


In [ ]:
question = "What is photosynthesis?"
context = "Photosynthesis is the process by which green plants use sunlight to produce energy."

inputs = bert_tokenizer(question, context, return_offsets_mapping=True, return_tensors="pt")
tokens = bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for tok, offset in zip(tokens, inputs["offset_mapping"][0]):
    print(f"{tok:15s} → {offset}")


The decoder-only model has no separate "question" and "context" inputs: everything is one text, read left to right. Here is exactly what Qwen3 receives for the same pair.

In [ ]:
messages = [{"role": "user", "content": f"Context: {context}\nQuestion: {question}"}]
qwen_prompt = qwen_tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
print(qwen_prompt)
print("Qwen3 tokens:", len(qwen_tokenizer(qwen_prompt)["input_ids"]),
      "| BERT tokens:", inputs["input_ids"].shape[1])

### 🧪 Exercise: Token Boundaries
1. What token separates question and context?
2. What does `[CLS]` and `[SEP]` represent?

---


In [ ]:
# Myabe Google it ?

## 🧠 Extractive QA with BERT

We'll manually run inference to extract the answer span from the context.

---


In [ ]:
# convert to tokens id + attention mask (to ignore padding )
inputs = bert_tokenizer(question, context, return_tensors="pt")
with torch.no_grad():
    outputs = bert_model(**inputs)
#finds the highest start token (when the answer starts)
start = torch.argmax(outputs.start_logits)
#finds the highest end token (when the answer ends)
end = torch.argmax(outputs.end_logits)
answer = bert_tokenizer.decode(inputs["input_ids"][0][start:end+1])
print("📘 BERT Extracted Answer:", answer)


### 🧪 Challenge: Use BERT on Your Own Paragraph

1. Write your own context (2–3 sentences).
2. Write a question.
3. Use BERT to extract the answer.

---


In [ ]:
#Start Your Work Here

## ✨ Generative QA with T5

We now feed the model a structured prompt and let it **generate** the answer.

---


In [ ]:
prompt = f"question: {question} context: {context}"
inputs = t5_tokenizer(prompt, return_tensors="pt")
outputs = t5_model.generate(**inputs)

print("📝 T5 Generated Answer:", t5_tokenizer.decode(outputs[0], skip_special_tokens=True))


### 🧪 Exercise: Prompt Design

1. Modify the prompt to make the question more detailed.
2. Try putting the context before the question.
3. Try removing the context — how does T5 behave?

---


In [ ]:
#Try it here

## Generative QA with Qwen3 (Decoder-Only)

The same question and context, now written by the decoder-only model.

---

In [ ]:
print("Qwen3 Generated Answer:", ask_qwen(question, context))

A decoder-only model is pretrained only to predict the next token. Without the chat template it simply continues the text, whatever pattern the text sets up. The chat template, learned in a later round of training on conversations, is what turns the continuation into an answer.

In [ ]:
raw = qwen_tokenizer("The capital of Saudi Arabia is", return_tensors="pt").to(device)
with torch.no_grad():
    continuation = qwen_model.generate(**raw, max_new_tokens=15, do_sample=False)
print("Plain continuation:", qwen_tokenizer.decode(continuation[0], skip_special_tokens=True))
print("Chat template:     ", ask_qwen("What is the capital of Saudi Arabia?"))

### Exercise: Prompt Design for the Decoder

1. Remove "using only the context" from the prompt in `ask_qwen`. Does the answer change?
2. Ask for the answer in Arabic.
3. Put the question before the context.

---

In [ ]:
#Try it here

## 🌍 Real-World QA Example

Let's use a paragraph from a news-like text and ask questions with both models.

---


In [ ]:
article = "Apple announced iPhone 17 on September 10, 2025. Shipping begins on September 20."

question = "When will iPhone 17 ship?"

# BERT
inputs = bert_tokenizer(question, article, return_tensors="pt")
with torch.no_grad():
    outputs = bert_model(**inputs)

start = torch.argmax(outputs.start_logits)
end = torch.argmax(outputs.end_logits) +1
print("📘 BERT Answer:", bert_tokenizer.decode(inputs["input_ids"][0][start:end]))

# T5
prompt = f"question: {question} context: {article}"
inputs = t5_tokenizer(prompt, return_tensors="pt")
outputs = t5_model.generate(**inputs)
print("📝 T5 Answer:", t5_tokenizer.decode(outputs[0], skip_special_tokens=True))

# Qwen3
print("Qwen3 Answer:", ask_qwen(question, article))

## 💣 Hallucination Test

We'll test what happens when the answer isn't present in the context.

---


In [ ]:
context = "Narnia is a fictional realm created by C.S. Lewis."
question = "What is the capital of Narnia?"

# BERT
inputs = bert_tokenizer(question, context, return_tensors="pt")
with torch.no_grad():
    outputs = bert_model(**inputs)

s = torch.argmax(outputs.start_logits)
e = torch.argmax(outputs.end_logits) + 1
print("📘 BERT:", bert_tokenizer.decode(inputs["input_ids"][0][s:e]))

# T5
prompt = f"question: {question} context: {context}"
inputs = t5_tokenizer(prompt, return_tensors="pt")
outputs = t5_model.generate(**inputs)
print("📝 T5:", t5_tokenizer.decode(outputs[0], skip_special_tokens=True))

# Qwen3
print("Qwen3:", ask_qwen(question, context))

## 🧾 T5 with Explanation

We now ask the model to justify its answer.

---


In [ ]:
q = "Why are plants green?"
ctx = "Chlorophyll in plants reflects green light and absorbs red/blue light."

prompt = f"question: {q} context: {ctx}\nExplain your answer."
inputs = t5_tokenizer(prompt, return_tensors="pt")
outputs = t5_model.generate(**inputs, max_length=256)

print(t5_tokenizer.decode(outputs[0], skip_special_tokens=True))

# Qwen3
print("Qwen3:", ask_qwen(q + " Explain your answer.", ctx, max_new_tokens=128))

## 🔢 Token Budget Visualization (BERT)

What happens when context exceeds 512 tokens?

---


In [ ]:
# BERT - Token Truncation
long_ctx = "foo " * 600 + "END. The secret code is 42."
question = "What is the secret code?"

bert_inputs = bert_tokenizer(question, long_ctx, return_tensors="pt", truncation=True)
print("🧮 BERT Token count:", bert_inputs["input_ids"].shape[1])

with torch.no_grad():
    outputs = bert_model(**bert_inputs)

s = torch.argmax(outputs.start_logits)
e = torch.argmax(outputs.end_logits) + 1
bert_answer = bert_tokenizer.decode(bert_inputs["input_ids"][0][s:e])
print("📘 BERT Answer:", bert_answer)

# T5 - Handles longer context
prompt = f"question: {question} context: {long_ctx}"
t5_inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True)
print("🧮 T5 Token count:", t5_inputs["input_ids"].shape[1])

t5_output = t5_model.generate(**t5_inputs, max_length=50)
print("📝 T5 Answer:", t5_tokenizer.decode(t5_output[0], skip_special_tokens=True))

# Qwen3 - its context window is far longer
qwen_tokens = len(qwen_tokenizer(long_ctx)["input_ids"])
print("Qwen3 Token count:", qwen_tokens, "| window:", qwen_model.config.max_position_embeddings)
print("Qwen3 Answer:", ask_qwen(question, long_ctx))

###✅ Interpretation
####Model	Observed Behavior	Why?
* BERT -->	Empty answer	The answer was beyond token 512, so it never reached it.
* T5   -->	Repetitive "foo" hallucination

* Qwen3 -->	Compare its token count with its window. The whole text fits, so the end of it is still there to read.

Every transformer has a window, set by its position embeddings and by the cost of attention, which grows with the square of the length. A longer window is a property of the model, not of the architecture alone.

## 🧠 Multiple Choice QA with T5

Let's let T5 decide between multiple options.

---


In [ ]:
# MCQA with T5
question = "Which planet is known as the Red Planet?"
choices = ["A) Earth", "B) Mars", "C) Jupiter", "D) Venus"]
prompt = question + "\n" + "\n".join(choices)

t5_inputs = t5_tokenizer(prompt, return_tensors="pt")
t5_output = t5_model.generate(**t5_inputs)
print("📝 T5 chose:", t5_tokenizer.decode(t5_output[0], skip_special_tokens=True))

# MCQA with BERT (simulate by searching each option in context) not fair tho
#try to remove the choices, lol
choices_str = " ".join(choices)
context =choices_str+" Mars is often called the Red Planet due to its reddish appearance."
bert_inputs = bert_tokenizer(question, context, return_tensors="pt")
with torch.no_grad():
    out = bert_model(**bert_inputs)
s = torch.argmax(out.start_logits)
e = torch.argmax(out.end_logits) + 1
bert_answer = bert_tokenizer.decode(bert_inputs["input_ids"][0][s:e])
print("📘 BERT span answer:", bert_answer)

# MCQA with Qwen3
print("Qwen3 chose:", ask_qwen(prompt + "\nAnswer with the letter only.", max_new_tokens=10))

## When the Answer Is Not in the Context

The paragraph mentions Germany but not its capital. An extractive model must still point somewhere in the paragraph. A generative model can write an answer it learned during pretraining, even when the prompt asks it to use only the context.

---

In [ ]:
question = "What is the capital of Germany?"
context = "Germany is a country in central Europe. It is known for its culture and science."

# BERT
bert_inputs = bert_tokenizer(question, context, return_tensors="pt")

with torch.no_grad():
    outputs = bert_model(**bert_inputs)

start = torch.argmax(outputs.start_logits)
end = torch.argmax(outputs.end_logits) + 1
bert_answer = bert_tokenizer.decode(bert_inputs["input_ids"][0][start:end])
print("[BERT Answer]:", bert_answer)

# T5
prompt = f"question: {question} context: {context}"
t5_inputs = t5_tokenizer(prompt, return_tensors="pt")
t5_output = t5_model.generate(**t5_inputs)
print("[T5 Answer]:", t5_tokenizer.decode(t5_output[0], skip_special_tokens=True))

# Qwen3
print("[Qwen3 Answer]:", ask_qwen(question, context))

## Summary: Which Architecture for Which Job

| | encoder-only (BERT) | encoder-decoder (T5) | decoder-only (Qwen3) |
|---|---|---|---|
| answer | a span copied from the text | written text | written text |
| can invent an answer | no | yes | yes |
| answer not in the text | still points at some span | often repeats words of the text | may say so, answer from pretraining, or invent an answer |
| window in this lab | 512 tokens | 512 tokens (as tokenized here) | tens of thousands of tokens |
| suits | finding facts in a document, classification | turning one text into another: translation, summaries | open questions, chat, following instructions |

Size matters as much as the architecture: the decoder here is several times larger than the other two models, and its answers cost more to compute.

---

# 🧪 Final Mini-Project Challenge

Build a mini QA system using BERT, T5 and Qwen3. Follow the steps:

---

### 🔶 Step 1: Choose or write a paragraph (4–5 sentences)
It could be about:
- A tech product (e.g., iPhone, Tesla)
- A scientific concept (e.g., gravity, DNA)
- A historical figure (e.g., Einstein, Cleopatra)

---

### 🔶 Step 2: Write 3 different questions
Each question should vary in:
- Type: Who/What/When/Why
- Difficulty: Easy → Tricky
- Presence of answer: One should have no answer

---

### 🔶 Step 3: Answer using BERT, T5 and Qwen3
Use manual inference cells to extract answers.
Then compare:
- Are they accurate?
- Did T5 or Qwen3 hallucinate?
- Did BERT miss or get cut off?

---

### 🔶 Step 4: Evaluation (Bonus)
If time permits, write:
- Which model was more reliable?
- Which handled ambiguity better?
- Any improvement suggestions?

In [ ]:
context = """
Ada Lovelace was an English mathematician and writer who lived in the 19th century.
She is known for her work on Charles Babbage's proposed mechanical general-purpose computer, the Analytical Engine.
Her notes on the engine include what is considered the first algorithm intended to be carried out by a machine.
Ada is often regarded as the first computer programmer.
"""


In [ ]:
questions = [
    "Who was Ada Lovelace?",
    "Why is Ada Lovelace considered the first programmer?",
    "What is Ada Lovelace's favorite color?"
]


In [ ]:
print("📘 BERT Answers\n" + "="*30)

for q in questions:
    inputs = bert_tokenizer(q, context, return_tensors="pt")
    with torch.no_grad():
        out = bert_model(**inputs)
    start = torch.argmax(out.start_logits)
    end = torch.argmax(out.end_logits) + 1
    answer = bert_tokenizer.decode(inputs["input_ids"][0][start:end])
    print(f"Q: {q}\n→ A: {answer}\n")


In [ ]:
print("📝 T5 Answers\n" + "="*30)

for q in questions:
    prompt = f"question: {q} context: {context}"
    inputs = t5_tokenizer(prompt, return_tensors="pt")
    outputs = t5_model.generate(**inputs)
    t5_answer = t5_tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Q: {q}\n→ A: {t5_answer}\n")


In [ ]:
print("Qwen3 Answers\n" + "="*30)

for q in questions:
    print(f"Q: {q}\n→ A: {ask_qwen(q, context)}\n")

## Exercises

### 1. A Bigger Decoder
Load `Qwen/Qwen3-1.7B` instead of `Qwen/Qwen3-0.6B` and rerun the hallucination test and the mini-project. Does a larger decoder invent less?

### 2. Where the Answer Sits
Build a context of about 400 words of filler with the answer sentence at the start, then at the end. Which models still find it in each case, and why?

### 3. The Same Question in Arabic
Ask all three models a question in Arabic about an Arabic paragraph. Which ones were trained on enough Arabic to answer?

---

## Further Reading

- Devlin et al., "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding," NAACL 2019. [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)
- Raffel et al., "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer," JMLR 2020. [arXiv:1910.10683](https://arxiv.org/abs/1910.10683)
- Qwen Team, "Qwen3 Technical Report," 2025. [arXiv:2505.09388](https://arxiv.org/abs/2505.09388)